# Taller práctico 2: Limpieza y análisis de datos con Pandas + Numpy 
### Contexto
Un equipo de investigadores está recopilando información sobre observaciones de animales realizadas en diferentes países de Europa Central y Oriental.

Los datos fueron recolectados por diferentes personas y presentan problemas de calidad: valores faltantes, registros duplicados, formatos inconsistentes, errores en los datos y diferentes formas de representar algunas variables.

Antes de realizar cualquier análisis, es necesario diagnosticar y limpiar la información.

Tu trabajo como analista de datos será preparar esta base para que pueda ser utilizada posteriormente en un análisis de biodiversidad.

### Dataset

Utiliza el dataset:

Animal Data — Dirty Data to Clean: What's Wrong with This Dataset?

Disponible en Kaggle:

https://www.kaggle.com/datasets/joannanplkrk/dirty-data-to-clean-whats-wrong-with-this-dataset

Descarga el archivo correspondiente a la versión dirty del dataset.

### 1. Diagnóstico inicial
Carga el dataset utilizando Pandas y realiza una exploración inicial.

Debes responder mediante código:

    • Cuántas filas y columnas tiene el DataFrame?
    • Cuáles son las columnas y qué tipo de dato tiene cada una?
    • Cuántos valores faltantes existen por columna?
    • Cuántos registros duplicados existen?
    • Utiliza unique() o value_counts() para identificar categorías inconsistentes o valores sospechosos.
Escribe una breve explicación de al menos 3 problemas de calidad que hayas encontrado.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

In [2]:
# Cargar el archivo
df = pd.read_csv('C:/Users/julya/OneDrive/Cursos2026/BIT_DataScience/DataBases/animal_data_dirty1.csv', sep=';')

In [3]:
df.shape

(1011, 11)

Podemos notar serias inconsitencias en los datos, por lo que hay que cambiar la forma en cómo se leen los datos del CSV usando como separador el ";"

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Animal type       991 non-null    str    
 1   Country           999 non-null    str    
 2   Weight kg         984 non-null    float64
 3   Body Length cm    984 non-null    float64
 4   Gender            992 non-null    str    
 5   Animal code       0 non-null      float64
 6   Latitude          913 non-null    float64
 7   Longitude         913 non-null    float64
 8   Animal name       52 non-null     str    
 9   Observation date  1011 non-null   str    
 10  Data compiled by  1011 non-null   str    
dtypes: float64(5), str(6)
memory usage: 87.0 KB


In [5]:
df.isnull().sum()

Animal type           20
Country               12
Weight kg             27
Body Length cm        27
Gender                19
Animal code         1011
Latitude              98
Longitude             98
Animal name          959
Observation date       0
Data compiled by       0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(167)

In [7]:
df_strings = df.select_dtypes(include='str')
for col in df_strings.columns:
    print(f"=== Frecuencia de valores en: {col} ===")
    print(df[col].value_counts(dropna=False)) 
    print("\n" + "-"*45 + "\n")

=== Frecuencia de valores en: Animal type ===
Animal type
red squirrel       543
hedgehog           274
lynx                61
European bison      49
red squirrell       22
NaN                 20
red squirel         14
lynx?               10
European bison™      6
European bisson      4
European buster      4
ledgehod             3
wedgehod             1
Name: count, dtype: int64

---------------------------------------------

=== Frecuencia de valores en: Country ===
Country
Poland            291
Germany           176
Slovakia          151
Hungary           146
Czech Republic    103
Austria            74
PL                 15
NaN                12
HU                 11
Hungry             10
CZ                  5
DE                  5
Czech               4
Australia           4
CC                  4
Name: count, dtype: int64

---------------------------------------------

=== Frecuencia de valores en: Gender ===
Gender
male              499
female            488
NaN                19
n

1. Cargar los datos del CSV con el separador correcto porque en este csv usaron ";".
2. La columna Observatrion date tiene valores que no son fechas, por lo que hay que convertirla a tipo fecha y manejar los errores.
3. La columna Animal code está vacía.
4. La columna Animal name tiene valores que no son nombres de animales, por lo que hay que limpiar esos valores.
5. La columna Animal type tiene valores que no son tipos de animales, por lo que hay que limpiar esos valores.
6. La columna Country tiene valores que no son países, por lo que hay que limpiar esos valores.

### 2. Limpieza de datos

Realiza las transformaciones necesarias para obtener un DataFrame consistente.

Debes:

    • Identificar las columnas con valores faltantes y aplica una estrategia de tratamiento adecuada.
    • Detectar y elimina los registros duplicados cuando corresponda.
    • Convertir las columnas que tengan tipos de datos incorrectos.
    • Identificar al menos dos variables categóricas que presenten inconsistencias y estandarízalas.
    • Después de realizar la limpieza, demuestra mediante código que las transformaciones fueron aplicadas correctamente.

In [8]:
df_clean = df.copy()

In [9]:
# Eliminamos las columnas Animal code y Animal name que no aportan información relevante y son en su mayoría valores nulos.
# Eliminamos duplicados. 
# Convertimos el formato de la fecha
# Estandarizar Country
# Reemplazar NaN de Country
# Estandarizar gender
# Reemplazar NaN de gender
# Estandarizar Animal type
# Reemplazar NaN de Animal type

country_map = {
    'PL': 'Poland',
    'DE': 'Germany',
    'HU': 'Hungary',
    'Hungry': 'Hungary',
    'CZ': 'Czech Republic',
    'Czech': 'Czech Republic',
    'CC': 'Czech Republic'
}

animal_map = {
    'red squirrell': 'Red Squirrel',
    'red squirel': 'Red Squirrel',
    'red squirrel': 'Red Squirrel',
    'lynx?': 'Lynx',
    'lynx': 'Lynx',
    'European bison™': 'European Bison',
    'European bisson': 'European Bison',
    'European buster': 'European Bison',
    'European bison': 'European Bison',
    'ledgehod': 'Hedgehog',
    'wedgehod': 'Hedgehog',
    'hedgehog': 'Hedgehog'
}

df_clean = (
    df
    .drop(columns=['Animal code', 'Animal name'])
    .drop_duplicates()
    .assign(**{
        'Observation date': lambda x: pd.to_datetime(
            x['Observation date'],
            format='mixed',
            dayfirst=True,
            errors='coerce'
        ),
        'Country': lambda x: (
            x['Country']
            .replace(country_map)
            .fillna('Unknown')
            .astype(str)
            .str.strip()
        ),
        'Gender': lambda x: x['Gender'].fillna('not determined').astype(str).str.strip().str.lower(),
        'Animal type': lambda x: (
            x['Animal type']
            .replace(animal_map)
            .fillna('Unknown')
            .astype(str)
            .str.strip()
        )
    })
    
    )



In [10]:
df_clean

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Observation date,Data compiled by
0,Unknown,Unknown,NaN,NaN,not determined,NaN,NaN,2024-01-03,James Johnson
1,Unknown,Unknown,NaN,NaN,not determined,NaN,NaN,2024-02-03,James Johnson
2,European Bison,Poland,930.000,335.0,male,52.828845,23.820144,2024-03-01,Anne Anthony
3,European Bison,Poland,909.000,311.0,not determined,52.830509,23.826849,2024-03-01,Anne Anthony
4,European Bison,Poland,581.000,277.0,female,52.834109,23.807093,2024-03-01,Anne Anthony
...,...,...,...,...,...,...,...,...,...
1005,Hedgehog,Hungary,0.900,23.0,male,47.510055,18.944356,2024-05-07,Anne Anthony
1006,Red Squirrel,Poland,0.346,20.0,female,52.212001,21.033187,2024-05-07,Anne Anthony
1007,Hedgehog,Germany,1.000,23.0,female,49.561356,11.105334,2024-05-07,Bob Bobson
1008,Hedgehog,Germany,0.500,17.0,female,49.561569,11.087046,2024-05-07,Bob Bobson


In [11]:
# Eliminar registros donde todos los campos numéricos y descriptivos principales sean nulos simultáneamente
df_clean = df_clean.dropna(
    subset=['Weight kg', 'Body Length cm', 'Latitude', 'Longitude'], 
    how='all'
)

In [12]:
# Imputación del peso según la mediana de cada especie
df_clean['Weight kg'] = df_clean.groupby('Animal type')['Weight kg'].transform(
    lambda group: group.fillna(group.median())
)

# Imputación de la longitud del cuerpo según la mediana de cada especie
df_clean['Body Length cm'] = df_clean.groupby('Animal type')['Body Length cm'].transform(
    lambda group: group.fillna(group.median())
)

In [13]:
df_clean['Weight kg'].describe()

count     833.000000
mean       46.898259
std       168.896354
min        -0.252000
25%         0.298000
50%         0.349000
75%         1.000000
max      1100.000000
Name: Weight kg, dtype: float64

In [14]:
df_clean['Body Length cm'].describe()

count    833.000000
mean      42.469388
std       63.110704
min      -19.000000
25%       19.000000
50%       21.000000
75%       23.000000
max      350.000000
Name: Body Length cm, dtype: float64

Al haber valores negativos, se decide poner su valor absoluto.

In [15]:
df_clean['Weight kg'] = df_clean['Weight kg'].abs()
df_clean['Body Length cm'] = df_clean['Body Length cm'].abs()

Demostración de las transformaciones aplicadas correctamente:

In [16]:
print("1. INFO GENERAL ")
print(df_clean.info())
print("-" * 50)

print("2. CONTEO DE NULOS")
print(df_clean.isnull().sum())
print("-" * 50)

print("3. FILAS DUPLICADAS")
print(f"Filas duplicadas en df_clean: {df_clean.duplicated().sum()}")
print("-" * 50)

print("4. VALORES ÚNICOS EN VARIABLES CATEGÓRICAS ESTANDARIZADAS")
print("Países:", df_clean['Country'].unique())
print("\nTipos de Animal:", df_clean['Animal type'].unique())
print("\nGéneros:", df_clean['Gender'].unique())
print("-" * 50)

print("5. VERIFICACIÓN DE MEDIDAS NUMÉRICAS POR ESPECIE")
print(df_clean.groupby('Animal type')[['Weight kg', 'Body Length cm']].describe())

1. INFO GENERAL 
<class 'pandas.DataFrame'>
Index: 833 entries, 2 to 1010
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Animal type       833 non-null    str           
 1   Country           833 non-null    str           
 2   Weight kg         833 non-null    float64       
 3   Body Length cm    833 non-null    float64       
 4   Gender            833 non-null    str           
 5   Latitude          752 non-null    float64       
 6   Longitude         752 non-null    float64       
 7   Observation date  833 non-null    datetime64[us]
 8   Data compiled by  833 non-null    str           
dtypes: datetime64[us](1), float64(4), str(4)
memory usage: 65.1 KB
None
--------------------------------------------------
2. CONTEO DE NULOS
Animal type          0
Country              0
Weight kg            0
Body Length cm       0
Gender               0
Latitude            81
Longitude          

In [17]:
print("Mínimo en Weight kg:", df_clean['Weight kg'].min())
print("Mínimo en Body Length cm:", df_clean['Body Length cm'].min())

Mínimo en Weight kg: 0.2
Mínimo en Body Length cm: 11.0


### 3. Numpy
Utiliza NumPy para realizar transformaciones sobre el DataFrame.

    • Crea una variable utilizando: np.where()
    La variable debe representar una condición relevante para el contexto de los datos.

    • Utiliza al menos una función estadística de NumPy para calcular una medida sobre una variable numérica.
        Puedes utilizar, por ejemplo:
        np.mean()
        np.median()
        np.nanmean()
        np.nanmedian()
        np.std()
        Explica brevemente qué representa el resultado obtenido.

In [18]:
# Condición relevante: Adulto por especie

umbral_q25 = df_clean.groupby('Animal type')['Weight kg'].transform(
    lambda x: np.percentile(x, 25)
)

In [19]:
df_clean['Adult'] = np.where(
    df_clean['Weight kg'] >= umbral_q25,
    'Yes',
    'No'
)

In [20]:
# Calculamos la mediana global del peso con Numpy
global_median_weight = np.median(df_clean['Weight kg'])

print(f"Mediana global del peso: {global_median_weight:.2f} kg")

Mediana global del peso: 0.35 kg


In [21]:
print(df_clean.groupby(['Animal type', 'Adult']).size())

Animal type     Adult
European Bison  No        16
                Yes       47
Hedgehog        No        36
                Yes      242
Lynx            No        15
                Yes       56
Red Squirrel    No       101
                Yes      312
Unknown         No         2
                Yes        6
dtype: int64


In [22]:
df_clean

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Observation date,Data compiled by,Adult
2,European Bison,Poland,930.000,335.0,male,52.828845,23.820144,2024-03-01,Anne Anthony,Yes
3,European Bison,Poland,909.000,311.0,not determined,52.830509,23.826849,2024-03-01,Anne Anthony,Yes
4,European Bison,Poland,581.000,277.0,female,52.834109,23.807093,2024-03-01,Anne Anthony,Yes
5,European Bison,Poland,900.000,295.0,male,52.834759,23.817201,2024-03-01,Anne Anthony,Yes
6,European Bison,Poland,620.000,250.0,female,52.834960,23.820750,2024-03-01,Anne Anthony,Yes
...,...,...,...,...,...,...,...,...,...,...
1005,Hedgehog,Hungary,0.900,23.0,male,47.510055,18.944356,2024-05-07,Anne Anthony,Yes
1006,Red Squirrel,Poland,0.346,20.0,female,52.212001,21.033187,2024-05-07,Anne Anthony,Yes
1007,Hedgehog,Germany,1.000,23.0,female,49.561356,11.105334,2024-05-07,Bob Bobson,Yes
1008,Hedgehog,Germany,0.500,17.0,female,49.561569,11.087046,2024-05-07,Bob Bobson,No


Se creó la columna Adult con np.where() para indicar si el animal es adulto o no, basándose en el peso del animal. Para evitar sesgos entre especies, la condición evalúa si el peso individual supera el primer cuartil de la respectiva especie.

Se utilizó np.percentile() para extrer el umbral del 25% del grupo y np.median() para conocer la tendencia central global del peso en la muestra.



### 4. Análisis con groupby()
Una vez que hayas terminado la limpieza, utiliza groupby() para responder preguntas sobre los datos.

Debes realizar al menos 4 análisis diferentes.

    • Cuál es el número de observaciones por tipo de animal?
    • Cuál es el promedio de una variable numérica según el tipo de animal?
    • Cuál es el promedio de una variable numérica según el país?

In [25]:
# Número de observaciones por tipo de animal

df_clean.groupby(['Animal type']).size()

Animal type
European Bison     63
Hedgehog          278
Lynx               71
Red Squirrel      413
Unknown             8
dtype: int64

In [28]:
# Promedio de una variable numérica según el tipo de animal

df_clean.groupby('Animal type')['Body Length cm'].mean()

Animal type
European Bison    244.238095
Hedgehog           20.205036
Lynx               82.901408
Red Squirrel       20.508475
Unknown            20.375000
Name: Body Length cm, dtype: float64

In [29]:
# Promedio de una variable numérica según el país

df_clean.groupby('Country')['Weight kg'].mean()

Country
Australia           0.494000
Austria             1.944041
Czech Republic     74.496190
Germany             0.456090
Hungary             2.882288
Poland            129.996125
Slovakia           32.449093
Name: Weight kg, dtype: float64

In [31]:
# Agrupar por país y tipo de animal para calcular el conteo de observaciones

df_clean.groupby(['Country', 'Animal type']).size()

Country         Animal type   
Australia       Hedgehog            2
                Red Squirrel        2
Austria         Hedgehog           25
                Lynx                5
                Red Squirrel       36
                Unknown             8
Czech Republic  European Bison     16
                Hedgehog           56
                Lynx                8
                Red Squirrel       36
Germany         Hedgehog           62
                Red Squirrel      116
Hungary         Hedgehog           56
                Lynx               13
                Red Squirrel       49
Poland          European Bison     38
                Hedgehog           21
                Lynx               19
                Red Squirrel      114
Slovakia        European Bison      9
                Hedgehog           56
                Lynx               26
                Red Squirrel       60
dtype: int64

### 5. Interpretación de resultados
Escribe al menos 4 conclusiones.

Tus conclusiones deben estar relacionadas con los datos.

Evita conclusiones genéricas como:

    "Los datos son interesantes."

Una buena conclusión debe indicar:

qué encontraste + evidencia + qué significa.

Por ejemplo:

    "El grupo X presenta el promedio más alto de , con un valor de . Esto indica que ___."

In [32]:
df_strings2 = df_clean.select_dtypes(include='str')
for col in df_strings2.columns:
    print(f"=== Frecuencia de valores en: {col} ===")
    print(df_clean[col].value_counts(dropna=False)) 
    print("\n" + "-"*45 + "\n")

=== Frecuencia de valores en: Animal type ===
Animal type
Red Squirrel      413
Hedgehog          278
Lynx               71
European Bison     63
Unknown             8
Name: count, dtype: int64

---------------------------------------------

=== Frecuencia de valores en: Country ===
Country
Poland            192
Germany           178
Slovakia          151
Hungary           118
Czech Republic    116
Austria            74
Australia           4
Name: count, dtype: int64

---------------------------------------------

=== Frecuencia de valores en: Gender ===
Gender
male              423
female            401
not determined      9
Name: count, dtype: int64

---------------------------------------------

=== Frecuencia de valores en: Data compiled by ===
Data compiled by
Anne Anthony     310
Bob Bobson       256
John Johnson     249
James Johnson     18
Name: count, dtype: int64

---------------------------------------------

=== Frecuencia de valores en: Adult ===
Adult
Yes    663
No     17

#### Conclusión 1:
El Red Squirrel es la especie más observada con presencia en todos los países del dataset con un valor de 413, lo que indica que es una especie común en las regiones.

#### Conclusión 2:
El European Bison es la especie de mayor tamaño y masa corporal en la muestra observada. Presenta el promedio de peso más alto de todo el dataset con 588 kg, superando por varios órdenes de magnitud a especies medianas o pequeñas como el Red Squirrel o el Hedgehog con promedios inferiores a 1 kg. Esto demuestra que el conjunto de datos es altamente heterogéneo y que cualquier análisis de métricas globales (como una media sin agrupar) estará fuertemente sesgado por los registros de bisontes.

#### Conclusión 3:
La gran mayoría de los registros corresponden a ejemplares desarrollados que superan el cuartil 25 de su respectiva especie. La variable condicional Adult mostró que 663 registros de un total de 844 (aproximadamente el 75%) quedaron clasificados dentro del rango de peso representativo de la especie. Esto sugiere que los datos recolectados corresponden predominantemente a individuos adultos, lo que reduce el sesgo de variabilidad de peso derivado de la presencia de crías en la muestra.

#### Conclusión 4:
Poland es el país con mayor volumen de observaciones registradas en el dataset, mientras que Australia representa una anomalía geográfica por el tipo de fauna. Poland lidera con 192 registros, seguido por Germany con 178 y Slovakia con 151. En contraste, Australia solo cuenta con 4 registros.

La recopilación de datos se concentra fuertemente en Europa Central. La presencia de 4 registros en Australia sugiere un error de captura o registros aislados fuera del rango de distribución natural de estas especies europeas.

#### Conclusión 5:
El trabajo de recolección y compilación de datos se encuentra distribuido principalmente entre tres investigadores principales. Anne Anthony compiló la mayor cantidad de datos con 310 registros, seguida por Bob Bobson (256) y John Johnson (249). En contraste, James Johnson solo aportó 18 registros. Esto demuestra que más del 97% de la recolección fue ejecutada por un equipo reducido de tres personas, lo que ayuda a mantener una consistencia metodológica en las observaciones del dataset.